<a href="https://colab.research.google.com/github/KrishDataLab/Machine_Learning_Mini_Projects/blob/main/LogisticRegression_Mini_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as plt

print("Import Models")

Import Models


# Task 1 — Load and Explore the Dataset

In [8]:
df = pd.read_csv('/content/churnguard_data.csv')
print(f"Original Shape: {df.shape}")

Original Shape: (1030, 12)


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1030 entries, 0 to 1029
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        1030 non-null   object 
 1   gender            1030 non-null   object 
 2   SeniorCitizen     1030 non-null   int64  
 3   tenure            980 non-null    float64
 4   PhoneService      1030 non-null   object 
 5   InternetService   1014 non-null   object 
 6   Contract          1030 non-null   object 
 7   PaperlessBilling  1030 non-null   object 
 8   PaymentMethod     1030 non-null   object 
 9   MonthlyCharges    955 non-null    float64
 10  TotalCharges      966 non-null    object 
 11  Churn             1030 non-null   object 
dtypes: float64(2), int64(1), object(9)
memory usage: 96.7+ KB


In [10]:
df.head()

,customerID,gender,SeniorCitizen,tenure,PhoneService,InternetService,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,CUST-0032,Male,0,21.0,YES,Fiber optic,Month-to-month,No,Credit card,29.73,600.01,Yes
1,CUST-0110,Male,0,55.0,YES,Fiber optic,Two year,Yes,Bank transfer,46.32,2515.48,No
2,CUST-0137,Female,1,46.0,Yes,Fiber optic,Month-to-month,No,Mailed check,87.06,4153.97,Yes
3,CUST-0089,Female,1,63.0,Yes,Fiber optic,Month-to-month,YES,Mailed check,56.97,3641.3,Yes
4,CUST-0919,Female,0,8.0,Yes,DSl,month to month,No,Electronic check,39.69,309.79,Yes


In [11]:
print(df.isnull().sum())

customerID           0
gender               0
SeniorCitizen        0
tenure              50
PhoneService         0
InternetService     16
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges      75
TotalCharges        64
Churn                0
dtype: int64


In [13]:
df.drop_duplicates()

,customerID,gender,SeniorCitizen,tenure,PhoneService,InternetService,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,CUST-0032,Male,0,21.0,YES,Fiber optic,Month-to-month,No,Credit card,29.73,600.01,Yes
1,CUST-0110,Male,0,55.0,YES,Fiber optic,Two year,Yes,Bank transfer,46.32,2515.48,No
2,CUST-0137,Female,1,46.0,Yes,Fiber optic,Month-to-month,No,Mailed check,87.06,4153.97,Yes
3,CUST-0089,Female,1,63.0,Yes,Fiber optic,Month-to-month,YES,Mailed check,56.97,3641.3,Yes
4,CUST-0919,Female,0,8.0,Yes,DSl,month to month,No,Electronic check,39.69,309.79,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...
1025,CUST-0088,Female,0,26.0,Yes,DSL,One year,Yes,Electronic check,51.68,1369.73,No
1026,CUST-0331,Male,0,49.0,Yes,Fiber optic,Two year,Yes,Electronic check,85.39,4305.4,No
1027,CUST-0467,Male,1,69.0,Yes,DSl,Two year,Yes,Electronic check,95.00,6689.62,No
1028,CUST-0122,Female,1,3.0,Yes,No,Two year,Yes,Electronic check,25.08,77.2,No


In [15]:
print(df['Churn'].value_counts())

Churn
No     598
Yes    278
NO      38
no      38
nO      27
yEs     20
YES     20
yes     11
Name: count, dtype: int64


In [16]:
print(df['Contract'].unique())

['Month-to-month' 'Two year' 'month to month' 'One year' 'month-to-month'
 'Monthly' 'Two Year' '1 year' '2 year' 'two year' 'One Year' 'one year']


# Task 2 — Clean the Dataset

In [18]:
# 1. Drop the customerID column
df = df.drop(columns=["customerID"])

In [23]:
df.columns

Index(['gender', 'SeniorCitizen', 'tenure', 'PhoneService', 'InternetService',
       'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges',
       'TotalCharges', 'Churn'],
      dtype='object')

In [28]:
# 2. Remove duplicate rows
df = df.drop_duplicates()

In [29]:
#3. Strip whitespace from gender and PaymentMethod using .str.strip()
df["gender"] = df["gender"].astype(str).str.strip()
df["PaymentMethod"] = df["PaymentMethod"].astype(str).str.strip()

In [30]:
# 4. Standardise casing — convert Churn, PhoneService, and PaperlessBilling to title case
for col in ["Churn", "PhoneService", "PaperlessBilling"]:
    df[col] = df[col].astype(str).str.strip().str.title()

In [33]:
# 5. Fix Contract — map all variations to valid values
contract_map = {
    "month to month": "Month-to-month",
        "month-to-month": "Month-to-month",
            "monthly": "Month-to-month",
                "one year": "One year",
                    "1 year": "One year",
                        "two year": "Two year",
                            "2 year": "Two year"
                            }
df["Contract"] = (
    df["Contract"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(contract_map)
        .fillna(df["Contract"])
)

In [36]:
# 6. Fix InternetService — map all variations to valid values
internet_map = {
     "dsl": "DSL",
     "fiber optic": "Fiber optic",
     "fibre optic": "Fiber optic",
     "fiberoptic": "Fiber optic",
     "no": "No",
     "none": "No",
     }
df["InternetService"] = (
 df["InternetService"]
               .astype(str)
               .str.strip()
               .str.lower()
               .map(internet_map)
               .fillna(df["InternetService"])
)




In [37]:
# 7. Fix TotalCharges — convert to numeric (junk becomes NaN)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [38]:
# 8. Remove rows where tenure is zero or negative
df = df[df["tenure"] > 0]

In [39]:
# 9. Remove rows where MonthlyCharges is less than 10 or greater than 200
df = df[(df["MonthlyCharges"] >= 10) & (df["MonthlyCharges"] <= 200)]

In [40]:
# 10. Fill missing values
# MonthlyCharges -> column mean
df["MonthlyCharges"] = df["MonthlyCharges"].fillna(df["MonthlyCharges"].mean())

# TotalCharges -> column mean
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].mean())

# tenure -> column median (rounded to integer)
df["tenure"] = df["tenure"].fillna(round(df["tenure"].median()))




In [41]:
# 11. Print the shape of the cleaned DataFrame
print("Cleaned DataFrame Shape:", df.shape)






Cleaned DataFrame Shape: (867, 11)


# 10.1 Fill remaining missing values in `InternetService`

Given that 'no' and 'none' are mapped to 'No' for `InternetService`, it is reasonable to assume that the remaining missing values also indicate a lack of internet service. Therefore, we will fill these `NaN`s with 'No'.

In [45]:
df['InternetService'] = df['InternetService'].fillna('No')

In [44]:
# 12. Print missing value counts to confirm all issues are resolved
print("\nMissing Value Counts:")
print(df.isnull().sum())


Missing Value Counts:
gender              0
SeniorCitizen       0
tenure              0
PhoneService        0
InternetService     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


# Task 3 — Train a Classification Model


In [46]:
# 2. Encode the target column Churn: Yes -> 1, No -> 0
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})







In [48]:
# 3. Encode categorical columns using pd.get_dummies() with drop_first=True
categorical_cols = [
    "gender",
     "PhoneService",
     "InternetService",
      "Contract",
      "PaperlessBilling",
       "PaymentMethod",
        ]
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [49]:
# 4. Separate features (X) and target (y)
X = df.drop(columns=["Churn"])
y = df["Churn"]

In [51]:
from sklearn.model_selection import train_test_split

# 5. Split into train and test sets (80% train, 20% test, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
    )

In [55]:
from sklearn.linear_model import LogisticRegression

# 6. Train a LogisticRegression model with max_iter=1000
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)






/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=1000)

In [59]:
from sklearn.metrics import accuracy_score

# Make predictions on the test set
y_pred = model.predict(X_test)
# 7. Print the accuracy score on the test set
print("Accuracy Score:", accuracy_score(y_test, y_pred))

Accuracy Score: 0.7011494252873564


In [61]:
from sklearn.metrics import classification_report

# 8. Print the classification report with target_names=['Stay', 'Churn']
print("\nClassification Report:")
print(
    classification_report(y_test, y_pred, target_names=["Stay", "Churn"])
            )


Classification Report:
              precision    recall  f1-score   support

        Stay       0.75      0.85      0.80       122
       Churn       0.50      0.35      0.41        52

    accuracy                           0.70       174
   macro avg       0.63      0.60      0.60       174
weighted avg       0.68      0.70      0.68       174



# Task 4 — Build a Prediction Script
The model is trained and evaluated. Now the business team wants a simple tool they can use to check whether a specific customer is at risk of churning — by entering that customer's details manually and getting an instant prediction.

In this task, you will retrain the model on the full cleaned dataset and build an interactive prediction script.

In [68]:
# 1. Encode Churn as Yes -> 1, No -> 0
# This step is redundant as it was already done in cell OKZe9R45auvc,
# and re-running it causes the Churn column to become all NaNs. Therefore, this line is removed.

In [64]:
# Select the features specified for Task 4, accounting for one-hot encoding
features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "SeniorCitizen",
    "Contract_One year",
    "Contract_Two year",
]
X = df[features]
y = df["Churn"]

In [ ]:
# 3. Retrain a LogisticRegression model on the full cleaned dataset
model = LogisticRegression(max_iter=1000)

# The temporary NaN handling is no longer needed after fixing the upstream encoding issue.
model.fit(X, y)

In [81]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

# 1. Re-load and clean churnguard_data.csv for Task 4
df = pd.read_csv('churnguard_data.csv')

# Apply all cleaning steps from Task 2 to ensure a fresh, clean DataFrame
if "customerID" in df.columns:
    df = df.drop(columns=["customerID"])

df = df.drop_duplicates()

df["gender"] = df["gender"].astype(str).str.strip()
df["PaymentMethod"] = df["PaymentMethod"].astype(str).str.strip()

for col in ["Churn", "PhoneService", "PaperlessBilling"]:
    df[col] = df[col].astype(str).str.strip().str.title()

contract_map = {
    "month to month": "Month-to-month",
    "month-to-month": "Month-to-month",
    "monthly": "Month-to-month",
    "one year": "One year",
    "1 year": "One year",
    "two year": "Two year",
    "2 year": "Two year",
}
df["Contract"] = (
    df["Contract"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(contract_map)
    .fillna(df["Contract"])
)

internet_map = {
    "dsl": "DSL",
    "fiber optic": "Fiber optic",
    "fibre optic": "Fiber optic",
    "fiberoptic": "Fiber optic",
    "no": "No",
    "none": "No",
}
df["InternetService"] = (
    df["InternetService"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(internet_map)
    .fillna(df["InternetService"])
)

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df[df["tenure"] > 0]
df = df[(df["MonthlyCharges"] >= 10) & (df["MonthlyCharges"] <= 200)]

df["MonthlyCharges"] = df["MonthlyCharges"].fillna(df["MonthlyCharges"].mean())
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].mean())
df["tenure"] = df["tenure"].fillna(round(df["tenure"].median()))
df['InternetService'] = df['InternetService'].fillna('No') # Fill remaining InternetService NaNs

# Specific mapping for 'Contract' to numeric for the 5-feature model (Task 4)
# This is crucial because 'Contract' is part of the 5 features and must be numeric.
contract_numeric_map = {
    "Month-to-month": 0,
    "One year": 1,
    "Two year": 2,
}
# Only apply if 'Contract' is still object type from the general cleaning
if df["Contract"].dtype == "object":
    df["Contract"] = df["Contract"].map(contract_numeric_map).fillna(0) # fillna(0) for any unmapped or NaN contracts

# 2. Encode Churn as Yes -> 1, No -> 0
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# Select the 5 features specified for Task 4
features = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen", "Contract"]
X = df[features]
y = df["Churn"]

# 3. Retrain a LogisticRegression model on the full cleaned dataset
model = LogisticRegression(max_iter=1000)
model.fit(X, y)

# 4. Use default values for prediction instead of collecting inputs manually
tenure = 12
monthly_charges = 50.0
total_charges = 600.0
senior_citizen = 0
contract = 0 # Month-to-month

# Build feature array for prediction
user_data = pd.DataFrame(
    [[tenure, monthly_charges, total_charges, senior_citizen, contract]],
    columns=features,
)

# 5. Use the trained model to predict churn
prediction = model.predict(user_data)[0]

# 6. Print the result
if prediction == 1:
    print("Prediction: This customer is likely to CHURN.")
else:
    print("Prediction: This customer is likely to STAY.")

Prediction: This customer is likely to STAY.
